# Laboratorio 5 — Clasificación con Árbol de Decisión
**Curso:** Minería de Datos (EIN132A25)
**Dataset:** Pokémon (Gen 1–6)

## Objetivos
- Preparar el dataset Pokémon para clasificación
- Entrenar un **árbol de decisión** para predecir si un Pokémon es **Legendario**
- Hacer predicciones y evaluar el modelo
- Visualizar el árbol e interpretar la importancia de features

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

url = "https://gist.githubusercontent.com/armgilles/194bcff35001e7eb53a2a8b441e8b2c6/raw/92200bc0a673d5ce2110aaad4544ed6c4010f687/pokemon.csv"
df = pd.read_csv(url)
print(f"Shape: {df.shape}")
df.head()

## 1. Preparar el dataset

**Target:** `Legendary` (True/False) — ¿es un Pokémon legendario?

**Features:** estadísticas de combate + generación + tipo principal

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Imputar Type 2
df["Type 2"] = df["Type 2"].fillna("None")

# Encoding de Type 1
le = LabelEncoder()
df["Type1_enc"] = le.fit_transform(df["Type 1"])

# Target: Legendary → int
df["Legendary_int"] = df["Legendary"].astype(int)

features = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed", "Generation", "Type1_enc"]
X = df[features]
y = df["Legendary_int"]

print(f"Shape de X: {X.shape}")
print(f"\nDistribución del target:")
print(y.value_counts().rename({0: 'No Legendario', 1: 'Legendario'}))
print(f"\nClase minoritaria: {y.mean()*100:.1f}% legendarios")

In [ ]:
# Visualizar distribución de stats por clase
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
stat_cols = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]

for i, col in enumerate(stat_cols):
    ax = axes[i // 3][i % 3]
    df[df["Legendary"] == False][col].hist(bins=20, alpha=0.6, color="steelblue", label="Normal", ax=ax)
    df[df["Legendary"] == True][col].hist(bins=20, alpha=0.6, color="gold", label="Legendario", ax=ax)
    ax.set_title(col)
    ax.legend()

plt.suptitle("Distribución de stats: Normal vs Legendario", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 2. Separar en Train y Test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]} muestras")
print(f"Test:  {X_test.shape[0]} muestras")
print(f"\nLegendarios en train: {y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"Legendarios en test:  {y_test.sum()} ({y_test.mean()*100:.1f}%)")

## 3. Entrenar el árbol de decisión

In [ ]:
modelo = DecisionTreeClassifier(max_depth=4, random_state=42)
modelo.fit(X_train, y_train)
print("Modelo entrenado.")
print(f"Profundidad del árbol: {modelo.get_depth()}")
print(f"Número de hojas: {modelo.get_n_leaves()}")

## 4. Predicciones y evaluación

In [ ]:
y_pred = modelo.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy en test: {accuracy:.4f} ({accuracy*100:.1f}%)")

In [ ]:
print(classification_report(y_test, y_pred, target_names=["No Legendario", "Legendario"]))

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Legendario", "Legendario"],
            yticklabels=["No Legendario", "Legendario"])
plt.title("Matriz de Confusión")
plt.ylabel("Real")
plt.xlabel("Predicho")
plt.tight_layout()
plt.show()

In [ ]:
acc_train = accuracy_score(y_train, modelo.predict(X_train))
acc_test  = accuracy_score(y_test, y_pred)
print(f"Accuracy TRAIN: {acc_train:.4f}")
print(f"Accuracy TEST:  {acc_test:.4f}")

## 5. Visualizar el árbol

In [ ]:
plt.figure(figsize=(22, 9))
plot_tree(modelo, feature_names=features,
          class_names=["Normal", "Legendario"],
          filled=True, rounded=True, fontsize=10)
plt.title("Árbol de Decisión — Clasificación de Pokémon Legendarios", fontsize=14)
plt.show()

## 6. Importancia de features

In [ ]:
importancias = pd.Series(modelo.feature_importances_, index=features).sort_values(ascending=False)
importancias.plot(kind="bar", color="gold", edgecolor="black")
plt.title("Importancia de cada feature — Árbol de Decisión")
plt.ylabel("Importancia")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
print(importancias)

## Ejercicios

### Ejercicio 1 — Comparar max_depth=1 vs max_depth=10

In [ ]:
for depth in [1, 10]:
    m = DecisionTreeClassifier(max_depth=depth, random_state=42)
    m.fit(X_train, y_train)
    tr = accuracy_score(y_train, m.predict(X_train))
    te = accuracy_score(y_test, m.predict(X_test))
    print(f"max_depth={depth:2d}: Train={tr:.4f}, Test={te:.4f}")

### Ejercicio 2 — Agregar Total como feature

In [ ]:
features2 = features + ["Total"]
X2 = df[features2]
X2_train, X2_test, _, _ = train_test_split(X2, y, test_size=0.2, random_state=42, stratify=y)
m2 = DecisionTreeClassifier(max_depth=4, random_state=42)
m2.fit(X2_train, y_train)
print(f"Accuracy SIN Total:  {accuracy_score(y_test, modelo.predict(X_test)):.4f}")
print(f"Accuracy CON Total:  {accuracy_score(y_test, m2.predict(X2_test)):.4f}")

### Desafío — Curva de overfitting

In [ ]:
depths = range(1, 16)
train_scores = []
test_scores  = []

for depth in depths:
    m = DecisionTreeClassifier(max_depth=depth, random_state=42)
    m.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, m.predict(X_train)))
    test_scores.append(accuracy_score(y_test, m.predict(X_test)))

plt.figure(figsize=(9, 5))
plt.plot(depths, train_scores, label="Train", marker="o", color="steelblue")
plt.plot(depths, test_scores, label="Test", marker="o", color="salmon")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Overfitting según profundidad — Dataset Pokémon")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()